In [ ]:
import pathlib
import ollama
from itertools import product

In [ ]:
OUTPUT = '../data/output.txt'
PROMPTS = '../prompts/prompts.txt'

In [ ]:
with open(OUTPUT) as input_file:
    text = f'{input_file.readlines()}'

text = text.strip('"')
text.replace('\'', '')

system_prompt = "Du bist ein erfahrener Rennfahrer-Coach. Du analysierst Fahrdaten und gibst präzises Feedback zu Linie, Bremspunkten, Gas/Bremse-Dosierung. Antworte kurz, fokussiert, praxisnah, maximal 2 Sätze. Timestamp ist immer in Sekunden. Jedes Attribut hat zwei Werte: Das Erste steht immer für die zu bewertende Runde, der zweite Wert beschreibt eine optimale Runde die dir als Referenz dient. **Verwende dafür ausschließlich die Daten aus der Liste des Users**. Negative Prompt: Denk dir keine weiteren Daten aus, Bewerte nicht die zweiten Werte der Attribute"
user_prompt = "Bewerte meine Fahrleistung, zeige mir klar die Unterschiede:"

# Just as an example, this would be the full set of laps and segments
# for lap, segment in [(0,0), (0,1), (0,2), (0,3), (0,4), (0,6), 
#                      (1,0), (1,1), (1,2), (1,3), (1,4),
#                      (2,0), (2,1), (2,2), (2,3), (2,4),]:

# list(product([0,1,2], [0,1,2,3,4]))

for lap, segment in [(0,1), (0,2)]:
    print(f'Lap: {lap}, Segment: {segment}')
    # Filter text for lap and segment
    filtered_lines = []
    for line in text.splitlines():
        if f"'lap_number': {lap}" in line and f"'segment': {segment}" in line:
            filtered_lines.append(line)
    filtered_text = '\n'.join(filtered_lines)

    # Generate LLM text from output file
    resp = ollama.chat(
        model="nemotron-3-nano:30b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + f"\n```json\n{filtered_text}\n```"},
        ],
    )
    print(resp["message"]["content"])

    # Get min and max timestamps from filtered_lines for logging
    min_timestamp, max_timestamp = float('inf'), float('-inf')
    for line in filtered_lines:
        timestamp = float(line.split("'timestamp': ")[1].split(',')[0])
        min_timestamp = min(min_timestamp, timestamp)
        max_timestamp = max(max_timestamp, timestamp)

    # Log prompts and responses
    pathlib.Path("prompts").mkdir(parents=True, exist_ok=True)
    with open(PROMPTS, "a", encoding="utf-8") as file:
        file.writelines([
            f"Lap: {lap}, Segment: {segment}, Sequence: {min_timestamp} - {max_timestamp}\n",
            "System Prompt: " + str(system_prompt) + "\n",
            "User Prompt: " + str(user_prompt) + "\n",
            "Response: " + str(resp["message"]["content"]) + "\n\n",
        ])



